In [62]:
import pandas as pd 
import numpy as np 
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [64]:
heartDisease = pd.read_csv('heart.csv').replace('?', np.nan).dropna()

In [66]:
for col in ['age', 'trestbps', 'chol', 'thalach']:
    heartDisease[col] = pd.cut(heartDisease[col] , bins=3, labels=['low', 'mid', 'high'])
heartDisease['target'] = (heartDisease['target'] > 0).astype(int)
print(heartDisease.head())

model = DiscreteBayesianNetwork([('age' , 'trestbps'), ('age', 'fbs'),('sex', 'trestbps'), ('exang', 'trestbps'),('trestbps', 'target'), 
                                 ('fbs', 'target'), ('target', 'restecg'),('target','thalach'), ('target','chol')])

mle = MaximumLikelihoodEstimator(model, heartDisease)
cpds = mle.get_parameters()
for cpd in cpds:
    model.add_cpds(cpd)

print('\nInterferencing:')
infer = VariableElimination(model)

q1 = infer.query(variables=['target'], evidence={'age' : 'low'})
print("P(Heart Disease | Age Low):") 
print(q1)

q2 = infer.query(variables=['target'], evidence={'chol': 'high'})
print()
print("P(Heart Disease | Chol High):")
print(q2)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'O', 'sex': 'N', 'cp': 'N', 'trestbps': 'O', 'chol': 'O', 'fbs': 'N', 'restecg': 'N', 'thalach': 'O', 'exang': 'N', 'oldpeak': 'N', 'slope': 'N', 'ca': 'N', 'thal': 'N', 'target': 'N'}


    age  sex  cp trestbps chol  fbs  restecg thalach  exang  oldpeak  slope  \
0  high    1   3      mid  low    1        0     mid      0      2.3      0   
1   low    1   2      mid  low    0        1    high      0      3.5      0   
2   low    0   1      mid  low    0        0    high      0      1.4      2   
3   mid    1   1      low  low    0        1    high      0      0.8      2   
4   mid    0   0      low  mid    0        1    high      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  

Interferencing:
P(Heart Disease | Age Low):
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.4413 |
+-----------+---------------+
| target(1) |        0.5587 |
+-----------+---------------+

P(Heart Disease | Chol High):
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |    